In [0]:
# ============================================================
# Silver — Source 14: Scrapy Competitor Pricing
#
# Transformations:
#   - Cast price_pence STRING to LONG (use try_cast)
#   - Validate our_price_pence > 0
#   - Normalise availability, currency
#   - Reject null product_sku or competitor_id → quarantine
#   - No timestamp — deduplicate on product_sku + competitor_id
#
# Source:  bronze.src_14_competitor.competitor_pricing
# Target:  silver.src_14_competitor.competitor_pricing
# Quarantine: silver.quarantine.src_14_competitor
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

BRONZE_CATALOG = 'bronze'
SILVER_CATALOG = 'silver'
TARGET_TABLE = f'{SILVER_CATALOG}.src_14_competitor.competitor_pricing'
QUARANTINE_TABLE = f'{SILVER_CATALOG}.quarantine.src_14_competitor'

VALID_AVAILABILITY = ['in_stock', 'out_of_stock', 'limited', 'preorder']

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {SILVER_CATALOG}.src_14_competitor')
print('Silver Source 14 Competitor Pricing — starting...')


In [0]:
bronze = spark.table(f'{BRONZE_CATALOG}.src_14_competitor.competitor_pricing')
total = bronze.count()
print(f'Bronze rows: {total}')

# Cast price_pence from STRING to LONG
df = bronze.withColumn('price_pence', F.expr('try_cast(price_pence as long)'))

# Normalise
df = df \
    .withColumn('product_sku',    F.upper(F.trim(F.col('product_sku')))) \
    .withColumn('currency',       F.upper(F.trim(F.col('currency')))) \
    .withColumn('availability',   F.lower(F.trim(F.col('availability'))))

# Bad rows
bad = df.filter(
    F.col('product_sku').isNull() |
    F.col('competitor_id').isNull() |
    F.col('our_price_pence').isNull() |
    (F.col('our_price_pence') <= 0)
).withColumn('quarantine_reason', F.lit('failed_validation')) \
 .withColumn('source_table', F.lit('competitor_pricing'))

# Good rows — dedup on product_sku + competitor_id
good = df.filter(
    F.col('product_sku').isNotNull() &
    F.col('competitor_id').isNotNull() &
    F.col('our_price_pence').isNotNull() &
    (F.col('our_price_pence') > 0)
).dropDuplicates(['product_sku', 'competitor_id'])

bad_count = bad.count()
good_count = good.count()
print(f'Competitor pricing: {total} total → {good_count} clean, {bad_count} quarantined ({bad_count/total*100:.1f}%)')

# Price comparison stats
print('\nPrice comparison (our price vs competitor):')
good.select(
    F.avg('price_difference_pct').alias('avg_diff_pct'),
    F.sum(F.when(F.col('price_difference_pct') < 0, 1).otherwise(0)).alias('we_are_cheaper'),
    F.sum(F.when(F.col('price_difference_pct') > 0, 1).otherwise(0)).alias('competitor_cheaper'),
    F.count('*').alias('total')
).show()

# Write
if spark.catalog.tableExists(TARGET_TABLE):
    dt = DeltaTable.forName(spark, TARGET_TABLE)
    dt.alias('t').merge(good.alias('s'), 't.product_sku = s.product_sku AND t.competitor_id = s.competitor_id') \
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    good.write.format('delta').mode('overwrite').saveAsTable(TARGET_TABLE)
print('✅ Written')

# Quarantine
if bad_count > 0:
    bad.select(
        F.lit('src_14_competitor').alias('source'),
        F.col('source_table'),
        F.col('quarantine_reason'),
        F.current_timestamp().alias('quarantined_at'),
        F.to_json(F.struct(*[c for c in bad.columns if c not in ['quarantine_reason','source_table']])).alias('raw_record')
    ).write.format('delta').mode('append').option('mergeSchema','true').saveAsTable(QUARANTINE_TABLE)
    print(f'✅ {bad_count} quarantined')


In [0]:
count = spark.sql(f'SELECT COUNT(*) as cnt FROM {TARGET_TABLE}').collect()[0]['cnt']
print(f'silver.src_14_competitor.competitor_pricing: {count} rows')
spark.sql(f"""
    SELECT competitor_name, COUNT(*) as products,
           ROUND(AVG(price_difference_pct),1) as avg_price_diff_pct
    FROM {TARGET_TABLE}
    GROUP BY competitor_name
    ORDER BY avg_price_diff_pct
""").show()
